In [1]:
# Import all the required libraries

import pdfplumber
from pathlib import Path
import pandas as pd
from operator import itemgetter
import json
import tiktoken
import openai
import chromadb

### Read and process the PDF File

In [3]:
pdf_file = 'Principal-Sample-Life-Insurance-Policy.pdf'

In [16]:
# Open the PDF file and print the text
with pdfplumber.open(pdf_file) as pdf:
    
    # Extract text from the first page
    print('---------- First Page -----------')
    single_page = pdf.pages[0]
    print(single_page)
    
    print('---------- Seventh Page -----------')
    # Get one of the pages from the PDF and examine it
    single_page = pdf.pages[5]
    text = single_page.extract_text()
    print(text)
    
    print('---------- Tables -----------')
    # Extract tables from the first page
    tables = single_page.extract_tables()
    print(tables)

---------- First Page -----------
<Page:1>
---------- Seventh Page -----------
TABLE OF CONTENTS
PART I - DEFINITIONS
PART II - POLICY ADMINISTRATION
Section A – Contract
Entire Contract Article 1
Policy Changes Article 2
Policyholder Eligibility Requirements Article 3
Policy Incontestability Article 4
Individual Incontestability Article 5
Information to be Furnished Article 6
Certificates Article 7
Assignments Article 8
Dependent Rights Article 9
Policy Interpretation Article 10
Electronic Transactions Article 11
Section B – Premium
Payment Responsibility; Due Dates; Grace Period Article 1
Premium Rates Article 2
Premium Rate Changes Article 3
Premium Amount Article 4
Contributions from Members Article 5
Section C - Policy Termination
Failure to Pay Premium Article 1
Termination Rights of the Policyholder Article 2
Termination Rights of The Principal Article 3
Policyholder Responsibility to Members Article 4
Section D - Policy Renewal
Renewal Article 1
PART III - INDIVIDUAL REQUIREMEN

In [18]:
# Function to check whether a word if present in a table for segregation of regular text and tables

def check_bboxes(word, table_bbox):
    # Check whether word is inside a table bbox.
    l = word['x0'], word['top'], word['x1'], word['bottom']
    r = table_bbox
    return l[0] > r[0] and l[1] > r[1] and l[2] < r[2] and l[3] < r[3]

#### Function to extract text from a PDF file.
- Declare a variable i to store the iteration of the loop that will help us store page numbers alongside the text
- Declare an empty list 'full_text' to store all the text files
- Use pdfplumber to open the pdf pages one by one
- Find the tables and their locations in the page
- Extract the text from the tables in the variable 'tables'
- Extract the regular words by calling the function check_bboxes() and checking whether words are present in the table or not
- Use the cluster_objects utility to cluster non-table and table words together so that they retain the same chronology as in the original PDF
- Declare an empty list 'lines' to store the page text
- If a text element in present in the cluster, append it to 'lines', else if a table element is present, append the table
- Append the page number and all lines to full_text, and increment 'i'
- When the function has iterated over all pages, return the 'full_text' list

In [25]:
def extract_text_from_pdf(pdf_file):
    i = 0
    full_text = []
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            page_no = f"Page {i+1}"
            text = page.extract_text()

            tables = page.find_tables()
            table_bboxes = [i.bbox for i in tables]
            tables = [{'table': i.extract(), 'top': i.bbox[1]} for i in tables]
            non_table_words = [word for word in page.extract_words() if not any(
                [check_bboxes(word, table_bbox) for table_bbox in table_bboxes])]
            lines = []

            for cluster in pdfplumber.utils.cluster_objects(non_table_words + tables, itemgetter('top'), tolerance=5):

                if 'text' in cluster[0]:
                    try:
                        lines.append(' '.join([i['text'] for i in cluster]))
                    except KeyError:
                        pass

                elif 'table' in cluster[0]:
                    lines.append(json.dumps(cluster[0]['table']))


            full_text.append([page_no, " ".join(lines)])
            i +=1

    return full_text

In [31]:
data = []

# Processing the PDF file
print(f"Processing... {pdf_file}")

# Call the function to extract the text from the PDF
extracted_text = extract_text_from_pdf(pdf_file)

# Convert the extracted list to a PDF, and add a column to store document names
extracted_text_df = pd.DataFrame(extracted_text, columns=['Page No.', 'Page_Text'])

# Append the extracted text and document name to the list
data.append(extracted_text_df)

# Print a message to indicate progress
print(f"Finished processing {pdf_file}")

# Print a message to indicate all PDFs have been processed
print("PDF have been processed.")

Processing... Principal-Sample-Life-Insurance-Policy.pdf
Finished processing Principal-Sample-Life-Insurance-Policy.pdf
PDF have been processed.


In [32]:
# print the text
data

[   Page No.                                          Page_Text
 0    Page 1  DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...
 1    Page 2                 This page left blank intentionally
 2    Page 3  POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...
 3    Page 4                 This page left blank intentionally
 4    Page 5  PRINCIPAL LIFE INSURANCE COMPANY (called The P...
 ..      ...                                                ...
 59  Page 60  I f a Dependent who was insured dies during th...
 60  Page 61  Section D - Claim Procedures Article 1 - Notic...
 61  Page 62  A claimant may request an appeal of a claim de...
 62  Page 63                 This page left blank intentionally
 63  Page 64  Principal Life Insurance Company Des Moines, I...
 
 [64 rows x 2 columns]]

In [35]:
# Concat the data
insurance_pdfs_data = pd.concat(data, ignore_index = True)

In [37]:
# Print Page No and Page_Text from the dataframe
insurance_pdfs_data

,Page No.,Page_Text
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...
1,Page 2,This page left blank intentionally
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...
3,Page 4,This page left blank intentionally
4,Page 5,PRINCIPAL LIFE INSURANCE COMPANY (called The P...
...,...,...
59,Page 60,I f a Dependent who was insured dies during th...
60,Page 61,Section D - Claim Procedures Article 1 - Notic...
61,Page 62,A claimant may request an appeal of a claim de...
62,Page 63,This page left blank intentionally


In [39]:
# Store the metadata for each page in a separate column
insurance_pdfs_data['Metadata'] = insurance_pdfs_data.apply(lambda x: {'Page_No.': x['Page No.']}, axis=1)
insurance_pdfs_data

,Page No.,Page_Text,Metadata
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,{'Page_No.': 'Page 1'}
1,Page 2,This page left blank intentionally,{'Page_No.': 'Page 2'}
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,{'Page_No.': 'Page 3'}
3,Page 4,This page left blank intentionally,{'Page_No.': 'Page 4'}
4,Page 5,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,{'Page_No.': 'Page 5'}
...,...,...,...
59,Page 60,I f a Dependent who was insured dies during th...,{'Page_No.': 'Page 60'}
60,Page 61,Section D - Claim Procedures Article 1 - Notic...,{'Page_No.': 'Page 61'}
61,Page 62,A claimant may request an appeal of a claim de...,{'Page_No.': 'Page 62'}
62,Page 63,This page left blank intentionally,{'Page_No.': 'Page 63'}


In [41]:
# Check one of the extracted page texts to ensure that the text has been correctly read
insurance_pdfs_data.Page_Text[2]

'POLICY RIDER GROUP INSURANCE POLICY NO: S655 COVERAGE: Life EMPLOYER: RHODE ISLAND JOHN DOE Effective on the later of the Date of Issue of this Group Policy or March 1, 2005, the following will apply to your Policy: From time to time The Principal may offer or provide certain employer groups who apply for coverage with The Principal a Financial Services Hotline and Grief Support Services or any other value added service for the employees of that employer group. In addition, The Principal may arrange for third party service providers (i.e., optometrists, health clubs), to provide discounted goods and services to those employer groups who apply for coverage with The Principal or who become insureds/enrollees of The Principal. While The Principal has arranged these goods, services and/or third party provider discounts, the third party service providers are liable to the applicants/insureds/enrollees for the provision of such goods and/or services. The Principal is not responsible for the

In [49]:
# Also check the length of all the texts as there might be some empty pages or pages with very few words that we can drop
insurance_pdfs_data['Text_Length'] = insurance_pdfs_data['Page_Text'].apply(lambda x: len(x.split(' ')))

In [51]:
# print the page length
insurance_pdfs_data['Text_Length']

0      30
1       5
2     230
3       5
4     110
     ... 
59    285
60    418
61    322
62      5
63      8
Name: Text_Length, Length: 64, dtype: int64

In [53]:
# Print the dataframe
insurance_pdfs_data

,Page No.,Page_Text,Metadata,Text_Length
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,{'Page_No.': 'Page 1'},30
1,Page 2,This page left blank intentionally,{'Page_No.': 'Page 2'},5
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,{'Page_No.': 'Page 3'},230
3,Page 4,This page left blank intentionally,{'Page_No.': 'Page 4'},5
4,Page 5,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,{'Page_No.': 'Page 5'},110
...,...,...,...,...
59,Page 60,I f a Dependent who was insured dies during th...,{'Page_No.': 'Page 60'},285
60,Page 61,Section D - Claim Procedures Article 1 - Notic...,{'Page_No.': 'Page 61'},418
61,Page 62,A claimant may request an appeal of a claim de...,{'Page_No.': 'Page 62'},322
62,Page 63,This page left blank intentionally,{'Page_No.': 'Page 63'},5


In [55]:
# Check the entire page's text
insurance_pdfs_data['Page_Text']

0     DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...
1                    This page left blank intentionally
2     POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...
3                    This page left blank intentionally
4     PRINCIPAL LIFE INSURANCE COMPANY (called The P...
                            ...                        
59    I f a Dependent who was insured dies during th...
60    Section D - Claim Procedures Article 1 - Notic...
61    A claimant may request an appeal of a claim de...
62                   This page left blank intentionally
63    Principal Life Insurance Company Des Moines, I...
Name: Page_Text, Length: 64, dtype: object

In [57]:
# Iterating over all page titles to create the final df with individual chunks
page_nos = insurance_pdfs_data["Page No."]
page_nos

0      Page 1
1      Page 2
2      Page 3
3      Page 4
4      Page 5
       ...   
59    Page 60
60    Page 61
61    Page 62
62    Page 63
63    Page 64
Name: Page No., Length: 64, dtype: object

In [61]:
# Function to split text into fixed-size chunks
def split_text_into_chunks(text, chunk_size):
    chunks = []
    words = text.split()  # Split the text into words

    current_chunk = []  # Store words for the current chunk
    current_chunk_word_count = 0  # Count of words in the current chunk

    for word in words:
        if current_chunk_word_count + len(word) + 1 <= chunk_size:
            current_chunk.append(word)
            current_chunk_word_count += len(word) + 1
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [word]
            current_chunk_word_count = len(word)

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks## 3.  Generating Embeddings 

In [63]:
def process_page(page_no):
    page = insurance_pdfs_data[insurance_pdfs_data['Page No.'] == page_no].Page_Text.values[0]
    metadata = insurance_pdfs_data[insurance_pdfs_data['Page No.'] == page_no].Metadata.values[0]

    if page is not None:
        # setting chunk size as 500
        chunk_size = 500
        text_chunks = split_text_into_chunks(page, chunk_size)

        # Creating a DataFrame to store the chunks, page title and page metadata
        data = {'Title': [], 'Chunk Text': [], 'Metadata': []}

        for index, chunk in enumerate(text_chunks):
            data['Title'].append(page_no)
            data['Chunk Text'].append(chunk)
            # adding chunk no as part of metadata
            metadata['Chunk_No.'] = index
            data['Metadata'].append(metadata)

        return pd.DataFrame(data)

In [65]:
# creating a dataframe after calling process
all_dfs = []
for page_no in page_nos:
    df = process_page(page_no)
    if df is not None:
        all_dfs.append(df)

fixed_chunk_df = pd.concat(all_dfs, ignore_index=True)
fixed_chunk_df

,Title,Chunk Text,Metadata
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,"{'Page_No.': 'Page 1', 'Chunk_No.': 0}"
1,Page 2,This page left blank intentionally,"{'Page_No.': 'Page 2', 'Chunk_No.': 0}"
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}"
3,Page 3,arrange for third party service providers (i.e...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}"
4,Page 3,the provision of such goods and/or services no...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}"
...,...,...,...
226,Page 62,"requested additional information, The Principa...","{'Page_No.': 'Page 62', 'Chunk_No.': 3}"
227,Page 62,may have the Member or Dependent whose loss is...,"{'Page_No.': 'Page 62', 'Chunk_No.': 3}"
228,Page 62,proof of loss has been filed and before the ap...,"{'Page_No.': 'Page 62', 'Chunk_No.': 3}"
229,Page 63,This page left blank intentionally,"{'Page_No.': 'Page 63', 'Chunk_No.': 0}"


### Generating Embeddings

In [70]:
# Import the SentenceTransformer library
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [71]:
# Load the embedding model
model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

In [72]:
# Function to generate embeddings for text
def generate_embeddings(texts):
    embeddings = embedder.encode(texts, convert_to_tensor=False)
    return embeddings

In [76]:
# Function to generate embedding on dataframe
def generate_embeddings_on_df(df):
  df['Embeddings'] = df['Chunk Text'].apply(lambda x: generate_embeddings([x])[0])

In [78]:
generate_embeddings_on_df(fixed_chunk_df)

In [79]:
# print the dataframe
fixed_chunk_df

,Title,Chunk Text,Metadata,Embeddings
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,"{'Page_No.': 'Page 1', 'Chunk_No.': 0}","[-0.025921917, 0.047777507, 0.055857766, 0.042..."
1,Page 2,This page left blank intentionally,"{'Page_No.': 'Page 2', 'Chunk_No.': 0}","[0.029118957, 0.060574066, 0.04641531, 0.03779..."
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}","[-0.0645379, 0.04319714, -8.394773e-05, -0.004..."
3,Page 3,arrange for third party service providers (i.e...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}","[-0.10200987, -0.028467676, -0.020565087, -0.0..."
4,Page 3,the provision of such goods and/or services no...,"{'Page_No.': 'Page 3', 'Chunk_No.': 2}","[-0.090008385, 0.07658207, 0.0049276487, -0.08..."
...,...,...,...,...
226,Page 62,"requested additional information, The Principa...","{'Page_No.': 'Page 62', 'Chunk_No.': 3}","[-0.04767192, 0.11277702, 0.06906491, -0.05086..."
227,Page 62,may have the Member or Dependent whose loss is...,"{'Page_No.': 'Page 62', 'Chunk_No.': 3}","[-0.079419956, 0.14404444, 0.031876147, -0.065..."
228,Page 62,proof of loss has been filed and before the ap...,"{'Page_No.': 'Page 62', 'Chunk_No.': 3}","[-0.1420601, 0.12368375, 0.120924726, -0.01142..."
229,Page 63,This page left blank intentionally,"{'Page_No.': 'Page 63', 'Chunk_No.': 0}","[0.029118957, 0.060574066, 0.04641531, 0.03779..."


### Store Embeddings in ChromaDB

In [83]:
import chromadb

# Call PersistentClient()
client = chromadb.PersistentClient(path = '/ChromaDB_Data/')

In [85]:
# Create a collection to store the embeddings. Collections in Chroma are where you can store your embeddings, documents, and any additional metadata.
collection = client.get_or_create_collection(name="insurance-collection")

In [87]:
collection.add(
    embeddings = fixed_chunk_df['Embeddings'].to_list(),
    documents = fixed_chunk_df['Chunk Text'].to_list(),
    metadatas = fixed_chunk_df['Metadata'].to_list(),
    ids = [str(i) for i in range(0, len(fixed_chunk_df['Embeddings']))]
)

Add of existing embedding ID: 0
Add of existing embedding ID: 1
Add of existing embedding ID: 2
Add of existing embedding ID: 3
Add of existing embedding ID: 4
Add of existing embedding ID: 5
Add of existing embedding ID: 6
Add of existing embedding ID: 7
Add of existing embedding ID: 8
Add of existing embedding ID: 9
Add of existing embedding ID: 10
Add of existing embedding ID: 11
Add of existing embedding ID: 12
Add of existing embedding ID: 13
Add of existing embedding ID: 14
Add of existing embedding ID: 15
Add of existing embedding ID: 16
Add of existing embedding ID: 17
Add of existing embedding ID: 18
Add of existing embedding ID: 19
Add of existing embedding ID: 20
Add of existing embedding ID: 21
Add of existing embedding ID: 22
Add of existing embedding ID: 23
Add of existing embedding ID: 24
Add of existing embedding ID: 25
Add of existing embedding ID: 26
Add of existing embedding ID: 27
Add of existing embedding ID: 28
Add of existing embedding ID: 29
Add of existing embe

In [89]:
# Get few of data by ids from collection
collection.get(
    ids = ['0','1','2'],
    include = ['embeddings', 'documents', 'metadatas']
)

{'ids': ['0', '1', '2'],
 'embeddings': array([[-2.59219166e-02,  4.77775075e-02,  5.58577664e-02, ...,
         -4.93265949e-02, -5.85114732e-02,  2.35519800e-02],
        [ 2.91189570e-02,  6.05740659e-02,  4.64153104e-02, ...,
          5.95401041e-02, -2.83836890e-02,  5.31937974e-03],
        [-6.45378977e-02,  4.31971401e-02, -8.39477289e-05, ...,
         -3.78734246e-02,  1.79674663e-02, -7.36598670e-03]]),
 'documents': ['DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/01/2014 711 HIGH STREET GEORGE RI 02903 GROUP POLICY FOR: RHODE ISLAND JOHN DOE ALL MEMBERS Group Member Life Insurance Print Date: 07/16/2014',
  'This page left blank intentionally',
  'POLICY RIDER GROUP INSURANCE POLICY NO: S655 COVERAGE: Life EMPLOYER: RHODE ISLAND JOHN DOE Effective on the later of the Date of Issue of this Group Policy or March 1, 2005, the following will apply to your Policy: From time to time The Principal may offer or provide certain employer groups who apply for coverage with The Princi

In [91]:
# Create a cache collection
cache_collection = client.get_or_create_collection(name='insurance-collection-cache')

In [93]:
# peek few of elements from cache collection
cache_collection.peek()

{'ids': ['what is the life insurance coverage for disability',
  'what is the Proof of ADL Disability or Total Disability',
  'what is condition of deatht while not wearing Seat Belt'],
 'embeddings': array([[-0.07620075,  0.10754956, -0.04078717, ..., -0.02822857,
          0.08649307,  0.08005349],
        [-0.1126289 ,  0.05175915, -0.00465672, ...,  0.10451437,
          0.02804654,  0.03974168],
        [ 0.00892728,  0.09783429, -0.02448488, ..., -0.0406705 ,
         -0.00775244,  0.04245879]]),
 'documents': ['what is the life insurance coverage for disability',
  'what is the Proof of ADL Disability or Total Disability',
  'what is condition of deatht while not wearing Seat Belt'],
 'uris': None,
 'data': None,
 'metadatas': [{'distances0': '0.7124123573303223',
   'distances1': '0.8027515411376953',
   'distances2': '0.8602005839347839',
   'distances3': '0.8729649186134338',
   'distances4': '0.8979799747467041',
   'distances5': '0.9084634184837341',
   'distances6': '0.910

### Semantic Search with Cache

In [104]:
# Read the user query
query = input()

 what is the life insurance coverage for disability


In [106]:
# Searh the Cache collection first
# Query the collection against the user query and return the top 20 results
cache_results = cache_collection.query(
    query_texts=query,
    n_results=1
)

In [108]:
# get result from cache collection
cache_results

{'ids': [['what is the life insurance coverage for disability']],
 'embeddings': None,
 'documents': [['what is the life insurance coverage for disability']],
 'uris': None,
 'data': None,
 'metadatas': [[{'distances0': '0.7124123573303223',
    'distances1': '0.8027515411376953',
    'distances2': '0.8602005839347839',
    'distances3': '0.8729649186134338',
    'distances4': '0.8979799747467041',
    'distances5': '0.9084634184837341',
    'distances6': '0.9102334976196289',
    'distances7': '0.9125920534133911',
    'documents0': 'Member Life Insurance or Coverage During Disability terminates under this Group Policy. This policy has been updated effective January 1, 2014 PART III - INDIVIDUAL REQUIREMENTS AND RIGHTS GC 6011 Section F - Individual Purchase Rights, Page 1',
    'documents1': "Payment of benefits will be subject to the Beneficiary and Facility of Payment provisions of this PART IV, Section A. Article 6 - Member Life Insurance - Coverage During Disability A Member may 

In [110]:
# get result from main collection
results = collection.query(
query_texts=query,
n_results=10
)
print("Result size is : " + str(len(results.items())))
results.items()

Result size is : 8


dict_items([('ids', [['143', '171', '147', '151', '183', '146', '180', '85', '179', '142']]), ('embeddings', None), ('documents', [['Member Life Insurance or Coverage During Disability terminates under this Group Policy. This policy has been updated effective January 1, 2014 PART III - INDIVIDUAL REQUIREMENTS AND RIGHTS GC 6011 Section F - Individual Purchase Rights, Page 1', "Payment of benefits will be subject to the Beneficiary and Facility of Payment provisions of this PART IV, Section A. Article 6 - Member Life Insurance - Coverage During Disability A Member may be eligible to continue his or her Member Life and Member Accidental Death and Dismemberment Insurance and Dependent Life Insurance coverage during the Member's ADL Disability or Total Disability. a. Coverage Qualification To be qualified for Coverage During Disability, a Member must: (1) become ADL", 'any Accelerated Benefit payment as described in PART IV, Section A, Article 7. Article 2 - Dependent Life Insurance a. Ind

In [112]:
# Implementing Cache in Semantic Search

# Set a threshold for cache search
threshold = 0.2

ids = []
documents = []
distances = []
metadatas = []
results_df = pd.DataFrame()


# If the distance is greater than the threshold, then return the results from the main collection.

if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
      # Query the collection against the user query and return the top 10 results
      results = collection.query(
      query_texts=query,
      n_results=10
      )

      # Store the query in cache_collection as document w.r.t to ChromaDB so that it can be embedded and searched against later
      # Store retrieved text, ids, distances and metadatas in cache_collection as metadatas, so that they can be fetched easily if a query indeed matches to a query in cache
      Keys = []
      Values = []

      size = len(results.items())

      for key, val in results.items():
        if val is None:
          continue
        for i in range(size):
          Keys.append(str(key)+str(i))
          Values.append(str(val[0][i]))


      cache_collection.add(
          documents= [query],
          ids = [query],  # Or if you want to assign integers as IDs 0,1,2,.., then you can use "len(cache_results['documents'])" as will return the no. of queries currently in the cache and assign the next digit to the new query."
          metadatas = dict(zip(Keys, Values))
      )

      print("Not found in cache. Found in main collection.")

      result_dict = {'Metadatas': results['metadatas'][0], 'Documents': results['documents'][0], 'Distances': results['distances'][0], "IDs":results["ids"][0]}
      results_df = pd.DataFrame.from_dict(result_dict)
      results_df


# If the distance is, however, less than the threshold, you can return the results from cache

elif cache_results['distances'][0][0] <= threshold:
      cache_result_dict = cache_results['metadatas'][0][0]

      # Loop through each inner list and then through the dictionary
      for key, value in cache_result_dict.items():
          if 'ids' in key:
              ids.append(value)
          elif 'documents' in key:
              documents.append(value)
          elif 'distances' in key:
              distances.append(value)
          elif 'metadatas' in key:
              metadatas.append(value)

      print("Found in cache!")

      # Create a DataFrame
      results_df = pd.DataFrame({
        'IDs': ids,
        'Documents': documents,
        'Distances': distances,
        'Metadatas': metadatas
      })

Found in cache!


In [114]:
# print the results
results_df

,IDs,Documents,Distances,Metadatas
0,143,Member Life Insurance or Coverage During Disab...,0.7124123573303223,"{'Chunk_No.': 4, 'Page_No.': 'Page 42'}"
1,171,Payment of benefits will be subject to the Ben...,0.8027515411376953,"{'Chunk_No.': 4, 'Page_No.': 'Page 49'}"
2,147,any Accelerated Benefit payment as described i...,0.8602005839347839,"{'Chunk_No.': 4, 'Page_No.': 'Page 43'}"
3,151,Dependent's Life Insurance terminates because ...,0.8729649186134338,"{'Chunk_No.': 4, 'Page_No.': 'Page 44'}"
4,183,disability that: (1) results from willful self...,0.8979799747467041,"{'Chunk_No.': 3, 'Page_No.': 'Page 51'}"
5,146,be the Coverage During Disability benefit in f...,0.9084634184837341,"{'Chunk_No.': 4, 'Page_No.': 'Page 43'}"
6,180,Total Disability began. Failure to give Writte...,0.9102334976196289,"{'Chunk_No.': 5, 'Page_No.': 'Page 50'}"
7,85,"terms of the Prior Policy, to have their premi...",0.9125920534133911,"{'Chunk_No.': 4, 'Page_No.': 'Page 28'}"


### Re-Ranking with a Cross Encoder

In [118]:
# Import the CrossEncoder library from sentence_transformers
from sentence_transformers import CrossEncoder

In [120]:
# Initialise the cross encoder model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [122]:
# Input (query, response) pairs for each of the top 20 responses received from the semantic search to the cross encoder
# Generate the cross_encoder scores for these pairs

cross_inputs = [[query, response] for response in results_df['Documents']]
cross_rerank_scores = cross_encoder.predict(cross_inputs)

In [124]:
# pritn the cross rerank scores
cross_rerank_scores

array([ 3.0535288 ,  4.810114  ,  0.5387689 ,  2.9317641 , -0.90366566,
        1.4724658 ,  0.26232514, -1.3536794 ], dtype=float32)

In [126]:
results_df['Reranked_scores'] = cross_rerank_scores

In [128]:
results_df

,IDs,Documents,Distances,Metadatas,Reranked_scores
0,143,Member Life Insurance or Coverage During Disab...,0.7124123573303223,"{'Chunk_No.': 4, 'Page_No.': 'Page 42'}",3.053529
1,171,Payment of benefits will be subject to the Ben...,0.8027515411376953,"{'Chunk_No.': 4, 'Page_No.': 'Page 49'}",4.810114
2,147,any Accelerated Benefit payment as described i...,0.8602005839347839,"{'Chunk_No.': 4, 'Page_No.': 'Page 43'}",0.538769
3,151,Dependent's Life Insurance terminates because ...,0.8729649186134338,"{'Chunk_No.': 4, 'Page_No.': 'Page 44'}",2.931764
4,183,disability that: (1) results from willful self...,0.8979799747467041,"{'Chunk_No.': 3, 'Page_No.': 'Page 51'}",-0.903666
5,146,be the Coverage During Disability benefit in f...,0.9084634184837341,"{'Chunk_No.': 4, 'Page_No.': 'Page 43'}",1.472466
6,180,Total Disability began. Failure to give Writte...,0.9102334976196289,"{'Chunk_No.': 5, 'Page_No.': 'Page 50'}",0.262325
7,85,"terms of the Prior Policy, to have their premi...",0.9125920534133911,"{'Chunk_No.': 4, 'Page_No.': 'Page 28'}",-1.353679


In [130]:
# Return the top 3 results from semantic search

top_3_semantic = results_df.sort_values(by='Distances')
top_3_semantic[:3]

,IDs,Documents,Distances,Metadatas,Reranked_scores
0,143,Member Life Insurance or Coverage During Disab...,0.7124123573303223,"{'Chunk_No.': 4, 'Page_No.': 'Page 42'}",3.053529
1,171,Payment of benefits will be subject to the Ben...,0.8027515411376953,"{'Chunk_No.': 4, 'Page_No.': 'Page 49'}",4.810114
2,147,any Accelerated Benefit payment as described i...,0.8602005839347839,"{'Chunk_No.': 4, 'Page_No.': 'Page 43'}",0.538769


In [132]:
# Return the top 3 results after reranking

top_3_rerank = results_df.sort_values(by='Reranked_scores', ascending=False)
top_3_rerank[:3]

,IDs,Documents,Distances,Metadatas,Reranked_scores
1,171,Payment of benefits will be subject to the Ben...,0.8027515411376953,"{'Chunk_No.': 4, 'Page_No.': 'Page 49'}",4.810114
0,143,Member Life Insurance or Coverage During Disab...,0.7124123573303223,"{'Chunk_No.': 4, 'Page_No.': 'Page 42'}",3.053529
3,151,Dependent's Life Insurance terminates because ...,0.8729649186134338,"{'Chunk_No.': 4, 'Page_No.': 'Page 44'}",2.931764


In [134]:
top_3_RAG = top_3_rerank[["Documents", "Metadatas"]][:3]
top_3_RAG

,Documents,Metadatas
1,Payment of benefits will be subject to the Ben...,"{'Chunk_No.': 4, 'Page_No.': 'Page 49'}"
0,Member Life Insurance or Coverage During Disab...,"{'Chunk_No.': 4, 'Page_No.': 'Page 42'}"
3,Dependent's Life Insurance terminates because ...,"{'Chunk_No.': 4, 'Page_No.': 'Page 44'}"


### Retrieval Augmented Generation

In [138]:
# Set the API key
with open("OpenAI_API_Key.txt", "r") as f:
  openai.api_key = ' '.join(f.readlines())

In [146]:
# Define the function to generate the response. Provide a comprehensive prompt that passes the user query and the top 3 results to the model

def generate_response(query, results_df):
    """
    Generate a response using GPT-3.5's ChatCompletion based on the user query and retrieved information.
    """
    messages = [
                {"role": "system", "content":  "You are a helpful assistant in the insurance domain who can effectively answer user queries about insurance policies and documents."},
                {"role": "user", "content": f"""You are a helpful assistant in the insurance domain who can effectively answer user queries about insurance policies and documents.
                                                You have a question asked by the user in '{query}' and you have some search results from a corpus of insurance documents in the dataframe '{top_3_RAG}'. These search results are essentially one page of an insurance document that may be relevant to the user query.

                                                The column 'documents' inside this dataframe contains the actual text from the policy document and the column 'metadata' contains the policy name and source page. The text inside the document may also contain tables in the format of a list of lists where each of the nested lists indicates a row.

                                                Use the documents in '{top_3_RAG}' to answer the query '{query}'. Frame an informative answer and also, use the dataframe to return the relevant policy names and page numbers as citations.

                                                Follow the guidelines below when performing the task.
                                                1. Try to provide relevant/accurate numbers if available.
                                                2. You don’t have to necessarily use all the information in the dataframe. Only choose information that is relevant.
                                                3. If the document text has tables with relevant information, please reformat the table and return the final information in a tabular in format.
                                                3. Use the Metadatas columns in the dataframe to retrieve and cite the policy name(s) and page numbers(s) as citation.
                                                4. If you can't provide the complete answer, please also provide any information that will help the user to search specific sections in the relevant cited documents.
                                                5. You are a customer facing assistant, so do not provide any information on internal workings, just answer the query directly.

                                                The generated response should answer the query directly addressing the user and avoiding additional information. If you think that the query is not relevant to the document, reply that the query is irrelevant. Provide the final response as a well-formatted and easily readable text along with the citation. Provide your complete response first with all information, and then provide the citations.
                                                """},
              ]

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    return response.choices[0].message.content.split('\n')

In [148]:
# Generate the response
response = generate_response(query, top_3_RAG)

In [149]:
# Print the response
print("\n".join(response))

In relation to life insurance coverage for disability, it is important to note that member life insurance generally covers death benefits; however, there might be specific provisions regarding coverage during periods of disability. 

The policy may provide that benefits will be subject to certain conditions when a member becomes disabled. Typically, this means that while the individual is disabled, they may be eligible for a different structure of benefits, which could include a continuation of their life insurance coverage or adjustments to the premiums.

Additionally, dependent life insurance usually terminates if the insured member becomes disabled, which could have implications for the overall coverage.

For more detailed information on these aspects, I would recommend reviewing the specific sections in your insurance documents that discuss life insurance coverage during disability. Below is a summarized overview of where to find relevant information:

| Policy Name                

### Queries

In [154]:
def search(query):

  # Set a threshold for cache search
  threshold = 0.2

  ids = []
  documents = []
  distances = []
  metadatas = []
  results_df = pd.DataFrame()

  # try to find from cache
  cache_results = cache_collection.query(
      query_texts=query,
      n_results=1
  )

  # If the distance is greater than the threshold, then return the results from the main collection.
  if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
        # Query the collection against the user query and return the top 10 results
        results = collection.query(
        query_texts=query,
        n_results=10
        )

        # Store the query in cache_collection as document w.r.t to ChromaDB so that it can be embedded and searched against later
        # Store retrieved text, ids, distances and metadatas in cache_collection as metadatas, so that they can be fetched easily if a query indeed matches to a query in cache
        Keys = []
        Values = []

        size = len(results.items())

        for key, val in results.items():
          if val is None:
            continue
          for i in range(size):
            Keys.append(str(key)+str(i))
            Values.append(str(val[0][i]))


        cache_collection.add(
            documents= [query],
            ids = [query],  # Or if you want to assign integers as IDs 0,1,2,.., then you can use "len(cache_results['documents'])" as will return the no. of queries currently in the cache and assign the next digit to the new query."
            metadatas = dict(zip(Keys, Values))
        )

        #print("Not found in cache. Found in main collection.")

        result_dict = {'Metadatas': results['metadatas'][0], 'Documents': results['documents'][0], 'Distances': results['distances'][0], "IDs":results["ids"][0]}
        results_df = pd.DataFrame.from_dict(result_dict)
        return results_df


  # If the distance is, however, less than the threshold, you can return the results from cache

  elif cache_results['distances'][0][0] <= threshold:
        cache_result_dict = cache_results['metadatas'][0][0]

        # Loop through each inner list and then through the dictionary
        for key, value in cache_result_dict.items():
            if 'ids' in key:
                ids.append(value)
            elif 'documents' in key:
                documents.append(value)
            elif 'distances' in key:
                distances.append(value)
            elif 'metadatas' in key:
                metadatas.append(value)

        #print("Found in cache!")

        # Create a DataFrame
        return pd.DataFrame({
          'IDs': ids,
          'Documents': documents,
          'Distances': distances,
          'Metadatas': metadatas
        })

In [156]:
def apply_cross_encoder(query, df):
  cross_inputs = [[query, response] for response in df['Documents']]
  cross_rerank_scores = cross_encoder.predict(cross_inputs)
  df['Reranked_scores'] = cross_rerank_scores
  return df

In [158]:
def get_topn(n, df):
  top_3_rerank = df.sort_values(by='Reranked_scores', ascending=False)
  return top_3_rerank[["Documents", "Metadatas"]][:n]

In [160]:
query = 'what is the life insurance coverage for disability'
df = search(query)
df = apply_cross_encoder(query, df)
df = get_topn(3, df)
response = generate_response(query, df)
print("\n".join(response))

Life insurance coverage for disability typically refers to the benefits that may be paid out if a policyholder becomes disabled. Based on the available documents, here are the relevant details:

1. **Member Life Insurance During Disability**: The policy provides information on how member life insurance coverage operates in the event of a disability. This can articulate the terms under which life insurance benefits are payable during periods of disability.

2. **Dependent's Life Insurance**: It's also noted that a dependent's life insurance can be affected if a member becomes disabled, which could lead to termination of coverage.

Unfortunately, the specific coverage amounts or percentages aren't detailed in the provided excerpts. For precise benefit amounts, you may want to refer to the relevant sections in the documents above.

Here is a summary of the references provided:

| Policy Name                                             | Page Number |
|-------------------------------------

In [161]:
query = 'what is the Proof of ADL Disability or Total Disability'
df = search(query)
df = apply_cross_encoder(query, df)
df = get_topn(3, df)
response = generate_response(query, df)
print("\n".join(response))

Proof of Activities of Daily Living (ADL) Disability or Total Disability typically involves providing specific documentation that demonstrates an individual's inability to engage in essential daily activities due to a medical condition or injury. Such proof often includes evaluations from qualified healthcare professionals, detailed medical records, and potentially assessments of the individual's ability to perform these activities safely.

The criteria for ADL disability generally focus on the individual's ability to perform the following tasks: 

1. **Bathing**: Ability to wash oneself.
2. **Dressing**: Ability to put on and take off clothing.
3. **Eating**: Ability to feed oneself.
4. **Toileting**: Ability to get to and from the toilet and manage personal hygiene.
5. **Transferring**: Ability to move from one position to another (such as from bed to a chair).
6. **Continence**: Ability to control bowel and bladder function.

It is crucial to consult the specific insurance policy fo

In [164]:
query = 'what is condition of deatht while not wearing Seat Belt'
df = search(query)
df = apply_cross_encoder(query, df)
df = get_topn(3, df)
response = generate_response(query, df)
print("\n".join(response))

The condition of death while not wearing a seat belt can often affect the benefits payable under an auto insurance policy, particularly in relation to life insurance and accidental deaths. Generally, if a policyholder dies in a vehicle accident while not wearing a seat belt, insurers may scrutinize the circumstances and determine whether the omission was a contributing factor to the fatality. This may lead to a reduction in the benefits or even a denial of the claim altogether.

To understand the specific implications under your insurance policy, please refer to the following sections from relevant documents:

| Policy Name                                              | Page Number |
|---------------------------------------------------------|-------------|
| Member Life Insurance or Coverage During Disability     | Page 42     |
| Payment of benefits will be subject to the Benefits clause | Page 49     |
| Dependent's Life Insurance                               | Page 44     |

These 